# NLP con Transformers y HuggingFace

En este notebook exploraremos el ecosistema de **HuggingFace** para procesamiento de lenguaje natural (NLP),
desde el uso de pipelines preentrenados hasta el fine-tuning de modelos BERT para tareas especificas.

**Objetivos:**
- Entender la tokenizacion y como los modelos procesan texto
- Usar pipelines de HuggingFace para tareas comunes de NLP
- Fine-tunear BERT para clasificacion de texto
- Trabajar con embeddings y similitud semantica

## Tokenizacion

Los modelos de lenguaje no trabajan directamente con texto, sino con **tokens**: unidades numericas
que representan fragmentos de texto.

### Tipos de tokenizacion

| Metodo | Ejemplo | Descripcion |
|--------|---------|-------------|
| **Por palabras** | `["Hello", "world"]` | Vocabulario enorme, no maneja palabras desconocidas |
| **Por caracteres** | `["H","e","l","l","o"]` | Vocabulario pequeno, secuencias muy largas |
| **Subword (BPE)** | `["Hel", "lo", "wor", "ld"]` | Balance entre ambos, usado en GPT |
| **WordPiece** | `["Hello", "##world"]` | Similar a BPE, usado en BERT |
| **SentencePiece** | `["_Hello", "_world"]` | Independiente del idioma, usado en T5 |

### Por que subword tokenization?
- Vocabulario manejable (30k-50k tokens)
- Maneja palabras nuevas o raras descomponiendolas
- Balancea longitud de secuencia vs granularidad
- Ejemplo: "unhappiness" -> `["un", "##happiness"]` (WordPiece)

### Tokens especiales
- `[CLS]`: Inicio de secuencia (usado para clasificacion)
- `[SEP]`: Separador entre segmentos
- `[PAD]`: Relleno para igualar longitudes
- `[MASK]`: Token enmascarado (para preentrenamiento MLM)
- `[UNK]`: Token desconocido

In [ ]:
# !pip install transformers datasets tokenizers torch

from transformers import AutoTokenizer

# Load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize a sample text
text = "HuggingFace Transformers is an amazing library for NLP!"

# Full tokenization output
encoded = tokenizer(text, return_tensors="pt", padding=True)

print(f"Texto original: {text}")
print(f"\nTokens: {tokenizer.tokenize(text)}")
print(f"\nInput IDs: {encoded['input_ids'].tolist()[0]}")
print(f"\nAttention mask: {encoded['attention_mask'].tolist()[0]}")
print(f"\nToken type IDs: {encoded['token_type_ids'].tolist()[0]}")

# Decode back to text
decoded = tokenizer.decode(encoded["input_ids"][0])
print(f"\nDecodificado: {decoded}")

# Show token-to-id mapping
print(f"\nMapeo token -> ID:")
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)
for token, token_id in zip(tokens, ids):
    print(f"  {token:<20} -> {token_id}")

# Vocabulary size
print(f"\nTamano del vocabulario: {tokenizer.vocab_size:,}")

# Special tokens
print(f"\nTokens especiales:")
print(f"  [CLS] ID: {tokenizer.cls_token_id}")
print(f"  [SEP] ID: {tokenizer.sep_token_id}")
print(f"  [PAD] ID: {tokenizer.pad_token_id}")
print(f"  [MASK] ID: {tokenizer.mask_token_id}")
print(f"  [UNK] ID: {tokenizer.unk_token_id}")

## HuggingFace Pipelines: NLP en 3 lineas

Los **pipelines** de HuggingFace son la forma mas rapida de usar modelos preentrenados.
Encapsulan todo el proceso: tokenizacion, inferencia y post-procesamiento.

Tareas disponibles:
- `sentiment-analysis`: Analisis de sentimiento
- `ner`: Reconocimiento de entidades nombradas
- `summarization`: Resumen de texto
- `translation`: Traduccion
- `question-answering`: Respuesta a preguntas
- `text-generation`: Generacion de texto
- `fill-mask`: Completar texto enmascarado
- `zero-shot-classification`: Clasificacion sin entrenamiento

In [ ]:
from transformers import pipeline

# --- Sentiment Analysis ---
print("=" * 50)
print("ANALISIS DE SENTIMIENTO")
print("=" * 50)

sentiment = pipeline("sentiment-analysis")
texts = [
    "I love this product, it's absolutely fantastic!",
    "This is terrible, worst purchase ever.",
    "The quality is okay, nothing special.",
]
for text, result in zip(texts, sentiment(texts)):
    print(f"  '{text[:50]}...' -> {result['label']} ({result['score']:.3f})")

# --- Named Entity Recognition ---
print(f"\n{'=' * 50}")
print("RECONOCIMIENTO DE ENTIDADES (NER)")
print("=" * 50)

ner = pipeline("ner", aggregation_strategy="simple")
ner_text = "Apple CEO Tim Cook announced new products at the event in San Francisco last Tuesday."
entities = ner(ner_text)
print(f"  Texto: '{ner_text}'\n")
for entity in entities:
    print(f"  {entity['word']:<20} -> {entity['entity_group']:<10} (score: {entity['score']:.3f})")

# --- Summarization ---
print(f"\n{'=' * 50}")
print("RESUMEN DE TEXTO")
print("=" * 50)

summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
article = """
Machine learning has transformed many industries in recent years. From healthcare to finance,
organizations are using ML models to automate decisions, predict outcomes, and discover patterns
in data. The rise of transformer models, starting with BERT in 2018, revolutionized natural
language processing. More recently, large language models like GPT have shown remarkable
capabilities in text generation, code writing, and reasoning tasks. However, challenges remain
in areas such as model interpretability, bias mitigation, and efficient deployment.
"""
summary = summarizer(article, max_length=60, min_length=20, do_sample=False)
print(f"  Original ({len(article.split())} palabras):")
print(f"  {article.strip()[:200]}...\n")
print(f"  Resumen ({len(summary[0]['summary_text'].split())} palabras):")
print(f"  {summary[0]['summary_text']}")

## Clasificacion de texto: fine-tuning BERT

Aunque los pipelines son utiles, a menudo necesitamos **fine-tunear** un modelo para nuestra tarea
especifica. Esto implica:

1. Cargar un modelo preentrenado (ej: BERT)
2. Anadir una capa de clasificacion encima
3. Entrenar con nuestros datos etiquetados

El fine-tuning es mucho mas rapido y requiere menos datos que entrenar desde cero, porque el modelo
ya "entiende" el lenguaje; solo necesita aprender nuestra tarea especifica.

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np

# Load IMDB dataset (movie review sentiment)
# Using a small subset for demonstration
dataset = load_dataset("imdb")

# Take a small subset for faster training
small_train = dataset["train"].shuffle(seed=42).select(range(1000))
small_test = dataset["test"].shuffle(seed=42).select(range(200))

print(f"Train samples: {len(small_train)}")
print(f"Test samples: {len(small_test)}")
print(f"\nLabels: {set(small_train['label'])} (0=negative, 1=positive)")

# Show class distribution
train_labels = small_train["label"]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["Negativo (0)", "Positivo (1)"],
       [train_labels.count(0), train_labels.count(1)],
       color=["#ff6b6b", "#51cf66"])
ax.set_ylabel("Cantidad")
ax.set_title("Distribucion de clases (IMDB subset)")
plt.tight_layout()
plt.show()

# Show example reviews
print("\nEjemplo de review POSITIVA:")
pos_idx = next(i for i, l in enumerate(small_train["label"]) if l == 1)
print(f"  {small_train['text'][pos_idx][:300]}...\n")

print("Ejemplo de review NEGATIVA:")
neg_idx = next(i for i, l in enumerate(small_train["label"]) if l == 0)
print(f"  {small_train['text'][neg_idx][:300]}...")

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

# Load tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization function
def tokenize_function(examples):
    """Tokenize text with truncation to max 512 tokens."""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,  # Shorter for faster training
    )

# Apply tokenization to the dataset using map()
tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test = small_test.map(tokenize_function, batched=True)

# Remove the original text column (model only needs input_ids, attention_mask)
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

# Rename label column to 'labels' (expected by HuggingFace Trainer)
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

# Set format to PyTorch tensors
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

# Data collator handles dynamic padding per batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"Columnas del dataset tokenizado: {tokenized_train.column_names}")
print(f"Ejemplo de input_ids shape: {tokenized_train[0]['input_ids'].shape}")
print(f"Ejemplo de attention_mask shape: {tokenized_train[0]['attention_mask'].shape}")

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Load pretrained model for sequence classification (2 classes)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)

# Define metrics computation
def compute_metrics(eval_pred):
    """Compute accuracy and F1 for evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted"),
    }

# Training configuration
training_args = TrainingArguments(
    output_dir="./results_imdb",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",  # Disable wandb/tensorboard
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Configuracion del Trainer:")
print(f"  Modelo: {model_name}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Parametros totales: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Parametros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
import matplotlib.pyplot as plt

# Train the model
train_result = trainer.train()

# Extract training metrics from log history
log_history = trainer.state.log_history

# Separate training and eval logs
train_losses = [(log["step"], log["loss"]) for log in log_history if "loss" in log and "eval_loss" not in log]
eval_metrics = [(log["step"], log.get("eval_loss"), log.get("eval_accuracy"), log.get("eval_f1"))
                for log in log_history if "eval_loss" in log]

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Training loss
if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, "b-", alpha=0.7)
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss")
    axes[0].grid(True, alpha=0.3)

# Eval loss
if eval_metrics:
    steps, eval_losses, accs, f1s = zip(*eval_metrics)
    axes[1].plot(steps, eval_losses, "r-o")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Validation Loss")
    axes[1].grid(True, alpha=0.3)

    # Eval accuracy and F1
    axes[2].plot(steps, accs, "g-o", label="Accuracy")
    axes[2].plot(steps, f1s, "m-s", label="F1")
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Score")
    axes[2].set_title("Validation Metrics")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

plt.suptitle("Curvas de entrenamiento - Fine-tuning DistilBERT en IMDB", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

# Evaluate on test set
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Classification report
print("Reporte de clasificacion:")
print("=" * 50)
print(classification_report(labels, preds, target_names=["Negativo", "Positivo"]))

# Confusion matrix
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Negativo", "Positivo"])
ax.set_yticklabels(["Negativo", "Positivo"])
ax.set_xlabel("Prediccion")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusion")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                fontsize=16, color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.colorbar(im)
plt.tight_layout()
plt.show()

# Show example predictions
print("\nEjemplos de predicciones:")
print("=" * 70)
test_texts = dataset["test"].shuffle(seed=42).select(range(200))["text"]
label_names = ["NEGATIVO", "POSITIVO"]

for i in range(5):
    text_preview = test_texts[i][:150]
    true_label = label_names[labels[i]]
    pred_label = label_names[preds[i]]
    match = "OK" if labels[i] == preds[i] else "ERROR"
    print(f"\n[{match}] Real: {true_label} | Pred: {pred_label}")
    print(f"  '{text_preview}...'")

## Embeddings y similitud semantica

Los **embeddings** son representaciones vectoriales densas del texto que capturan su significado
semantico. Dos textos con significado similar tendran embeddings cercanos en el espacio vectorial.

### Aplicaciones
- **Busqueda semantica**: Encontrar documentos relevantes
- **Clustering**: Agrupar textos similares
- **Deteccion de duplicados**: Identificar contenido repetido
- **RAG (Retrieval Augmented Generation)**: Base de sistemas de chat con documentos
- **Recomendaciones**: Recomendar contenido similar

In [ ]:
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import matplotlib.pyplot as plt

# Load a sentence embedding model
st_model = SentenceTransformer("all-MiniLM-L6-v2")

# Sentences to compare
sentences = [
    "The cat sat on the mat.",
    "A kitten was resting on the rug.",
    "Machine learning is a subset of artificial intelligence.",
    "AI and ML are closely related fields.",
    "The weather today is sunny and warm.",
    "It's a beautiful day with clear skies.",
    "Python is great for data science.",
    "I love cooking Italian pasta.",
]

# Compute embeddings
embeddings = st_model.encode(sentences)
print(f"Embedding shape: {embeddings.shape}")
print(f"  - {len(sentences)} sentences")
print(f"  - {embeddings.shape[1]} dimensions per embedding")

# Compute pairwise cosine similarity
sim_matrix = cosine_similarity(embeddings)

# Visualize similarity matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap="YlOrRd", vmin=0, vmax=1)

# Add labels
short_labels = [s[:35] + "..." if len(s) > 35 else s for s in sentences]
ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)

# Add similarity values
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center", fontsize=7)

ax.set_title("Matriz de similitud coseno entre oraciones", fontsize=14)
plt.colorbar(im, label="Similitud coseno")
plt.tight_layout()
plt.show()

# Find most similar pairs (excluding self-similarity)
print("\nPares mas similares:")
print("=" * 70)
pairs = []
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        pairs.append((sim_matrix[i, j], sentences[i], sentences[j]))

pairs.sort(key=lambda x: x[0], reverse=True)
for sim, s1, s2 in pairs[:5]:
    print(f"\n  Similitud: {sim:.4f}")
    print(f"  A: '{s1}'")
    print(f"  B: '{s2}'")

## Resumen: cuando fine-tunear vs usar pipeline vs usar LLM

| Enfoque | Cuando usarlo | Ventajas | Desventajas |
|---------|--------------|----------|-------------|
| **Pipeline preentrenado** | Tarea estandar (sentimiento, NER, resumen) sin datos propios | Rapido, sin entrenamiento, buena calidad general | No personalizable, puede no adaptarse a tu dominio |
| **Fine-tuning** | Tienes datos etiquetados (>500 ejemplos), tarea especifica, necesitas maxima precision | Alta precision en tu dominio, modelo dedicado, rapido en inferencia | Requiere datos, GPU, y expertise en ML |
| **LLM con prompting** | Prototipo rapido, pocos datos, tarea compleja que requiere razonamiento | Flexible, sin entrenamiento, maneja tareas complejas | Mas lento, mas caro por inferencia, menos predecible |
| **LLM + few-shot** | Tarea especifica pero pocos datos etiquetados | Mejor que zero-shot, sin entrenamiento | Limitado por context window, costo por token |

### Regla general

1. **Empieza con un pipeline** o LLM para validar que la tarea es viable
2. Si necesitas mas precision -> **fine-tunea** un modelo especifico
3. Si el costo de inferencia es critico -> fine-tunea un modelo pequeno (DistilBERT)
4. Si necesitas razonamiento complejo -> usa un **LLM grande** con prompting cuidadoso